# 01. 남양주시 방문자 수 추이 분석

## 분석 목적

본 노트북은 `방문자 수(연인원) 추이` 데이터를 활용하여 남양주시의 월별 관광 수요 규모와 전년 동월 대비 변화 흐름을 분석한다.

분석 결과는 모델 2인 `recommend_score_prediction`에서 관광지 추천 점수를 보정하는 월별 관광 수요 변수로 활용한다.

### 핵심 분석 질문

1. 남양주시 방문자 수는 월별로 어떤 흐름을 보이는가?
2. 전년 동월 대비 방문자 수가 증가하거나 감소한 월은 언제인가?
3. 분기별 관광 수요는 어떤 차이를 보이는가?
4. 추천 점수 예측 모델에 사용할 수 있는 파생변수는 무엇인가?


In [ ]:
# 라이브러리 불러오기
import os
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.preprocessing import MinMaxScaler

# 그래프 한글 표시 설정
# 실행 환경에 설치된 한글 폰트를 자동으로 선택한다.
font_candidates = ['NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR', 'AppleGothic', 'Malgun Gothic']
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}

for font in font_candidates:
    if font in installed_fonts:
        plt.rcParams['font.family'] = font
        break

plt.rcParams['axes.unicode_minus'] = False
print('사용 폰트:', plt.rcParams['font.family'])


## 1. 데이터 불러오기

파일명이 한글 조합형/완성형 문제로 정확히 매칭되지 않을 수 있으므로, `unicodedata.normalize()`를 이용해 파일명을 탐색한다.


In [ ]:
# 데이터 파일 경로 설정
DATA_DIR = Path('/mnt/data')
TARGET_FILE_NAME = '20260602152817_방문자 수(연인원) 추이.csv'

csv_path = None
for file in DATA_DIR.glob('*.csv'):
    if unicodedata.normalize('NFC', file.name) == TARGET_FILE_NAME:
        csv_path = file
        break

if csv_path is None:
    raise FileNotFoundError(f'파일을 찾을 수 없습니다: {TARGET_FILE_NAME}')

print('사용 파일:', csv_path)

visitor_df = pd.read_csv(csv_path)
visitor_df.head()


In [ ]:
# 데이터 기본 정보 확인
print('데이터 크기:', visitor_df.shape)
print('
컬럼 목록:')
print(visitor_df.columns.tolist())
print('
결측치 개수:')
print(visitor_df.isna().sum())

visitor_df.info()


## 2. 데이터 전처리

`기준년월`을 날짜형으로 변환하고, 분석에 필요한 연도, 월, 분기 컬럼을 생성한다.


In [ ]:
# 기준년월 전처리
visitor_df['기준년월'] = visitor_df['기준년월'].astype(str)
visitor_df['date'] = pd.to_datetime(visitor_df['기준년월'], format='%Y%m')
visitor_df['year'] = visitor_df['date'].dt.year
visitor_df['month'] = visitor_df['date'].dt.month
visitor_df['year_month'] = visitor_df['date'].dt.strftime('%Y-%m')
visitor_df['quarter'] = visitor_df['date'].dt.quarter

# 숫자형 컬럼 정리
numeric_cols = ['방문자수', '전년동월방문자수', '방문자수증감률']
for col in numeric_cols:
    visitor_df[col] = pd.to_numeric(visitor_df[col], errors='coerce')

visitor_df


## 3. 기본 통계 분석

총 방문자 수, 월평균 방문자 수, 최대 방문 월, 최소 방문 월 등을 확인한다.


In [ ]:
# 기본 통계 요약
total_visitors = visitor_df['방문자수'].sum()
total_prev_visitors = visitor_df['전년동월방문자수'].sum()
avg_visitors = visitor_df['방문자수'].mean()
median_visitors = visitor_df['방문자수'].median()
max_row = visitor_df.loc[visitor_df['방문자수'].idxmax()]
min_row = visitor_df.loc[visitor_df['방문자수'].idxmin()]
max_growth_row = visitor_df.loc[visitor_df['방문자수증감률'].idxmax()]
min_growth_row = visitor_df.loc[visitor_df['방문자수증감률'].idxmin()]
overall_growth_rate = ((total_visitors - total_prev_visitors) / total_prev_visitors) * 100

summary_df = pd.DataFrame({
    '항목': [
        '총 방문자 수',
        '전년 동월 기준 총 방문자 수',
        '전체 증감률',
        '월평균 방문자 수',
        '월중앙값 방문자 수',
        '최대 방문 월',
        '최소 방문 월',
        '최대 증감률 월',
        '최소 증감률 월'
    ],
    '값': [
        f'{total_visitors:,.0f}명',
        f'{total_prev_visitors:,.0f}명',
        f'{overall_growth_rate:.2f}%',
        f'{avg_visitors:,.0f}명',
        f'{median_visitors:,.0f}명',
        f"{max_row['year_month']} / {max_row['방문자수']:,.0f}명",
        f"{min_row['year_month']} / {min_row['방문자수']:,.0f}명",
        f"{max_growth_row['year_month']} / {max_growth_row['방문자수증감률']:.1f}%",
        f"{min_growth_row['year_month']} / {min_growth_row['방문자수증감률']:.1f}%"
    ]
})

summary_df


## 4. 월별 방문자 수 추이 시각화

월별 방문자 수 흐름을 선그래프로 확인한다. 이 그래프는 남양주시 관광 수요가 어느 시점에 높아지고 낮아지는지를 파악하기 위한 기본 시각화이다.


In [ ]:
# 결과 저장 폴더 생성
OUTPUT_DIR = Path('/mnt/data/visitor_trend_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1) 월별 방문자 수 추이
plt.figure(figsize=(12, 6))
plt.plot(visitor_df['year_month'], visitor_df['방문자수'], marker='o', label='방문자 수')
plt.title('남양주시 월별 방문자 수 추이')
plt.xlabel('기준년월')
plt.ylabel('방문자 수(명)')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_monthly_visitor_trend.png', dpi=300, bbox_inches='tight')
plt.show()


## 5. 방문자 수와 전년 동월 방문자 수 비교

현재 방문자 수와 전년 동월 방문자 수를 함께 시각화하여 전년 대비 수요 변화가 발생한 월을 확인한다.


In [ ]:
# 2) 방문자 수와 전년 동월 방문자 수 비교
plt.figure(figsize=(12, 6))
plt.plot(visitor_df['year_month'], visitor_df['방문자수'], marker='o', label='방문자 수')
plt.plot(visitor_df['year_month'], visitor_df['전년동월방문자수'], marker='s', label='전년 동월 방문자 수')
plt.title('방문자 수와 전년 동월 방문자 수 비교')
plt.xlabel('기준년월')
plt.ylabel('방문자 수(명)')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_visitor_vs_previous_year.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. 방문자 수 증감률 시각화

전년 동월 대비 증감률을 막대그래프로 확인한다. 양수는 전년 대비 증가, 음수는 전년 대비 감소를 의미한다.


In [ ]:
# 3) 전년 동월 대비 방문자 수 증감률
plt.figure(figsize=(12, 6))
plt.bar(visitor_df['year_month'], visitor_df['방문자수증감률'], label='방문자 수 증감률')
plt.axhline(0, linewidth=1)
plt.title('전년 동월 대비 방문자 수 증감률')
plt.xlabel('기준년월')
plt.ylabel('증감률(%)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_visitor_growth_rate.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. 분기별 방문자 수 분석

월별 데이터를 분기 단위로 묶어 계절적 수요 흐름을 확인한다.


In [ ]:
# 분기별 집계
quarter_df = visitor_df.groupby('quarter', as_index=False).agg(
    방문자수=('방문자수', 'sum'),
    전년동월방문자수=('전년동월방문자수', 'sum')
)

quarter_df['분기'] = quarter_df['quarter'].astype(str) + '분기'
quarter_df['분기별증감률'] = (
    (quarter_df['방문자수'] - quarter_df['전년동월방문자수'])
    / quarter_df['전년동월방문자수']
    * 100
)

quarter_df


In [ ]:
# 4) 분기별 방문자 수
plt.figure(figsize=(9, 6))
plt.bar(quarter_df['분기'], quarter_df['방문자수'], label='방문자 수')
plt.title('분기별 방문자 수')
plt.xlabel('분기')
plt.ylabel('방문자 수(명)')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_quarterly_visitor_count.png', dpi=300, bbox_inches='tight')
plt.show()


## 8. 모델 입력용 파생변수 생성

방문자 수와 방문자 수 증감률을 정규화한 뒤, 두 값을 결합하여 월별 관광 수요 점수인 `visitor_demand_score`를 생성한다.

- `visitor_count_scaled`: 방문자 수 정규화 값
- `visitor_growth_scaled`: 방문자 수 증감률 정규화 값
- `visitor_demand_score`: 월별 관광 수요 점수
- `is_peak_month`: 방문자 수 기준 상위 25% 월 여부


In [ ]:
# 정규화 및 모델용 파생변수 생성
scaler = MinMaxScaler()

visitor_df[['visitor_count_scaled']] = scaler.fit_transform(visitor_df[['방문자수']])
visitor_df[['visitor_growth_scaled']] = scaler.fit_transform(visitor_df[['방문자수증감률']])

visitor_df['visitor_demand_score'] = (
    visitor_df['visitor_count_scaled'] * 0.7
    + visitor_df['visitor_growth_scaled'] * 0.3
)

peak_threshold = visitor_df['방문자수'].quantile(0.75)
visitor_df['is_peak_month'] = (visitor_df['방문자수'] >= peak_threshold).astype(int)

model_feature_df = visitor_df[[
    'year_month',
    'year',
    'month',
    'quarter',
    '방문자수',
    '전년동월방문자수',
    '방문자수증감률',
    'visitor_count_scaled',
    'visitor_growth_scaled',
    'visitor_demand_score',
    'is_peak_month'
]].copy()

model_feature_df


In [ ]:
# 5) 월별 관광 수요 점수 시각화
plt.figure(figsize=(12, 6))
plt.plot(model_feature_df['year_month'], model_feature_df['visitor_demand_score'], marker='o', label='관광 수요 점수')
plt.title('월별 관광 수요 점수(visitor_demand_score)')
plt.xlabel('기준년월')
plt.ylabel('관광 수요 점수')
plt.xticks(rotation=45)
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_visitor_demand_score.png', dpi=300, bbox_inches='tight')
plt.show()


## 9. 분석 결과 저장

모델 2에서 바로 사용할 수 있도록 파생변수가 포함된 CSV 파일을 저장한다.


In [ ]:
# 분석 결과 저장
output_csv_path = OUTPUT_DIR / 'visitor_trend_features.csv'
model_feature_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print('저장 완료:', output_csv_path)
print('시각화 저장 폴더:', OUTPUT_DIR)


## 10. 분석 해석

방문자 수 추이 데이터는 남양주시의 월별 관광 수요 규모와 전년 동월 대비 변화 흐름을 보여준다. 분석 결과를 통해 방문자 수가 많은 성수기 월과 수요가 감소한 월을 구분할 수 있으며, 이는 추천 점수 예측 모델에서 월별 관광 수요를 보정하는 근거로 활용할 수 있다.

본 분석에서는 방문자 수와 방문자 수 증감률을 각각 정규화한 뒤, 방문자 수 규모에 70%, 증감률에 30%의 가중치를 부여하여 `visitor_demand_score`를 생성하였다. 이 변수는 모델 2에서 관광지 추천 점수를 계산할 때 남양주시 전체 관광 수요 흐름을 반영하는 보조 feature로 사용할 수 있다.
